# Multi-Omics for Drug Discovery
### Transcriptomics · Genomics · Proteomics · scRNA-seq · Integration · MOFA+

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

## The multi-omics vision

A drug perturbs cells at multiple levels simultaneously. Single-omics sees only one layer:

```
DNA mutations  →  Gene expression  →  Protein abundance  →  Metabolites
  (Genomics)      (Transcriptomics)     (Proteomics)       (Metabolomics)
      ↓                  ↓                   ↓                   ↓
  Resistance         Drug response        Drug target         Toxicity
  mechanisms         biomarkers           activity            biomarkers
                                ↓
                         Multi-omics integration
                        (MOFA+, SNF, deep learning)
                                ↓
                      Mechanistic drug understanding
```

| Section | Omics layer | Tools |
|---------|-------------|-------|
| 1. RNA-seq | Transcriptomics | PyDESeq2, GSEA, volcano plots |
| 2. scRNA-seq | Single-cell RNA | Scanpy, UMAP, cell typing |
| 3. Genomics | SNVs, CNVs, signatures | COSMIC, SBS, mutation landscapes |
| 4. Proteomics | Protein abundance | Differential expression, PPI |
| 5. MOFA+ | Integration | Multi-omics factor analysis |
| 6. Drug response | Phenotype prediction | GDSC, TCGA, survival models |
| 7. Toxicogenomics | Tox-specific | PathwayTox, DILI signatures |

---
## Section 1 — RNA-Seq & Differential Expression

In [ ]:
# ── Install ───────────────────────────────────────────────────────────────────
# !pip install pydeseq2 gseapy scanpy anndata matplotlib seaborn statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings; warnings.filterwarnings('ignore')
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
print("Imports OK ✓")

In [ ]:
# ── 1.1 Simulate an RNA-seq count matrix ─────────────────────────────────────
# In practice: use pydeseq2.DeseqDataSet with your count matrix and metadata

def simulate_rnaseq(n_genes: int = 5000, n_samples_per_group: int = 6,
                    n_de_genes: int = 400, seed: int = 42) -> tuple:
    """
    Simulate RNA-seq count matrix with DE genes.

    Returns
    -------
    counts     : DataFrame [n_genes × n_samples]  raw read counts
    metadata   : DataFrame [n_samples × 2]         condition + batch
    de_genes   : list of true DE gene names
    """
    rng = np.random.default_rng(seed)
    n_samples = n_samples_per_group * 2   # 2 groups

    gene_names   = [f"Gene_{i:04d}" for i in range(n_genes)]
    sample_names = ([f"CTRL_{i+1}" for i in range(n_samples_per_group)] +
                    [f"TREAT_{i+1}" for i in range(n_samples_per_group)])

    # Base expression: log-normal (realistic)
    base_expr = rng.lognormal(mean=5.0, sigma=1.5, size=n_genes)

    counts = np.zeros((n_genes, n_samples), dtype=int)
    de_genes = gene_names[:n_de_genes]

    for j, name in enumerate(sample_names):
        treated = name.startswith("TREAT")
        for i in range(n_genes):
            mu = base_expr[i]
            if treated and gene_names[i] in de_genes:
                # Upregulated: genes 0-199, downregulated: 200-399
                fc = 3.0 if i < n_de_genes//2 else 0.33
                mu *= fc
            # Negative-binomial noise (realistic for RNA-seq)
            mu_noisy = max(1, mu * rng.uniform(0.8, 1.2))
            counts[i, j] = rng.negative_binomial(5, 5/(5+mu_noisy))

    counts_df = pd.DataFrame(counts, index=gene_names, columns=sample_names)
    metadata  = pd.DataFrame({
        "condition": (["control"] * n_samples_per_group +
                      ["treated"] * n_samples_per_group),
        "batch":     ["batch1"]*3 + ["batch2"]*3 + ["batch1"]*3 + ["batch2"]*3,
    }, index=sample_names)

    return counts_df, metadata, de_genes

counts, metadata, true_de = simulate_rnaseq()
print(f"Count matrix: {counts.shape}  ({counts.shape[0]} genes × {counts.shape[1]} samples)")
print(f"\nMetadata:\n{metadata}")
print(f"\nTrue DE genes: {len(true_de)}  (first 5: {true_de[:5]})")
print(f"\nCount stats:")
print(f"  Min: {counts.values.min()}  Max: {counts.values.max()}")
print(f"  Mean: {counts.values.mean():.1f}  Zeros: {(counts.values==0).mean()*100:.1f}%")

In [ ]:
# ── 1.2 DESeq2-style differential expression ─────────────────────────────────
# Full pydeseq2: from pydeseq2 import DeseqDataSet, DeseqStats
# We implement the key normalisation + Wald test pipeline manually here

def normalise_counts(counts_df: pd.DataFrame) -> pd.DataFrame:
    """
    Median-of-ratios normalisation (DESeq2 method).
    Corrects for library size and composition differences.
    """
    # Geometric mean per gene (across samples)
    log_counts = np.log(counts_df.replace(0, np.nan))
    geom_mean  = log_counts.mean(axis=1)

    # Size factors: median ratio of each sample to geometric mean
    log_ratios  = log_counts.subtract(geom_mean, axis=0)
    size_factors = np.exp(log_ratios.median(axis=0))

    print(f"Size factors: {size_factors.round(3).to_dict()}")
    return counts_df.divide(size_factors, axis=1)

def de_analysis(counts_df: pd.DataFrame,
                metadata: pd.DataFrame,
                group_col: str = "condition",
                test: str = "ttest") -> pd.DataFrame:
    """
    Differential expression analysis.

    Returns DataFrame with log2FC, p-value, adjusted p-value, significance.
    """
    norm = normalise_counts(counts_df)
    ctrl_cols  = metadata[metadata[group_col]=="control"].index
    treat_cols = metadata[metadata[group_col]=="treated"].index

    results = []
    for gene in norm.index:
        ctrl_vals  = np.log2(norm.loc[gene, ctrl_cols]  + 1)
        treat_vals = np.log2(norm.loc[gene, treat_cols] + 1)
        log2fc     = treat_vals.mean() - ctrl_vals.mean()
        _, pval    = stats.ttest_ind(treat_vals, ctrl_vals)
        basemean   = norm.loc[gene].mean()
        results.append({"gene": gene, "log2FC": log2fc,
                        "pval": pval, "baseMean": basemean})

    df = pd.DataFrame(results)
    # Benjamini-Hochberg FDR correction
    from statsmodels.stats.multitest import multipletests
    _, padj, _, _ = multipletests(df["pval"].fillna(1), method="fdr_bh")
    df["padj"]    = padj
    df["sig"]     = (df["padj"] < 0.05) & (df["log2FC"].abs() > 1)
    df["direction"]= df.apply(lambda r: "up" if r.log2FC>1 else ("down" if r.log2FC<-1 else "ns"),
                               axis=1)
    return df.sort_values("padj").reset_index(drop=True)

de_results = de_analysis(counts, metadata)
print(f"\nDE results shape: {de_results.shape}")
print(f"Significant genes: {de_results['sig'].sum()}")
print(f"  Upregulated:   {(de_results['direction']=='up').sum()}")
print(f"  Downregulated: {(de_results['direction']=='down').sum()}")
print(f"\nTop 10 DE genes:")
print(de_results[de_results["sig"]].head(10)[["gene","log2FC","padj","direction"]].to_string())

In [ ]:
# ── 1.3 Volcano plot + heatmap ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Volcano plot ──────────────────────────────────────────────────────────────
ax = axes[0]
ns   = de_results[de_results["direction"]=="ns"]
up   = de_results[de_results["direction"]=="up"]
down = de_results[de_results["direction"]=="down"]

ax.scatter(ns["log2FC"],   -np.log10(ns["padj"]+1e-300),   c="#AAAAAA", s=5, alpha=0.4, label="ns")
ax.scatter(up["log2FC"],   -np.log10(up["padj"]+1e-300),   c="#E74C3C", s=8, alpha=0.7, label=f"Up ({len(up)})")
ax.scatter(down["log2FC"], -np.log10(down["padj"]+1e-300),  c="#1565C0", s=8, alpha=0.7, label=f"Down ({len(down)})")

# Label top genes
for _, row in de_results[de_results["sig"]].head(8).iterrows():
    ax.annotate(row["gene"], xy=(row["log2FC"], -np.log10(row["padj"]+1e-300)),
                fontsize=7, alpha=0.8)

ax.axvline(-1, color="k", linestyle="--", lw=1, alpha=0.4)
ax.axvline( 1, color="k", linestyle="--", lw=1, alpha=0.4)
ax.axhline(-np.log10(0.05), color="k", linestyle="--", lw=1, alpha=0.4)
ax.set_xlabel("log₂ Fold Change", fontsize=12)
ax.set_ylabel("-log₁₀(adj p-value)", fontsize=12)
ax.set_title("Volcano Plot — Drug Treatment vs Control", fontsize=13, fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.25)

# ── Top DE gene heatmap ───────────────────────────────────────────────────────
ax = axes[1]
top_genes = de_results[de_results["sig"]].head(30)["gene"].tolist()
hmap_data = np.log2(counts.loc[top_genes] + 1)
# Z-score across samples for each gene
hmap_z    = hmap_data.subtract(hmap_data.mean(axis=1), axis=0).divide(
             hmap_data.std(axis=1).replace(0, 1), axis=0)

im = ax.imshow(hmap_z.values, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3)
ax.set_xticks(range(hmap_z.shape[1]))
ax.set_xticklabels(hmap_z.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(top_genes)))
ax.set_yticklabels(top_genes, fontsize=7)
ax.set_title("Top 30 DE Genes — Z-score", fontsize=12, fontweight="bold")
plt.colorbar(im, ax=ax, label="Z-score")

plt.tight_layout(); plt.show()
print("Volcano and heatmap plotted ✓")

---
## Section 2 — Single-Cell RNA-seq

scRNA-seq gives per-cell resolution — revealing cell subpopulations, drug resistance clones, and TME composition.

In [ ]:
# ── 2.1 Scanpy workflow (simulated) ──────────────────────────────────────────
# Full workflow: import scanpy as sc; adata = sc.read_10x_mtx(...)
# We simulate an AnnData-like object here

try:
    import scanpy as sc
    import anndata as ad
    SCANPY = True
except ImportError:
    SCANPY = False
    print("scanpy not installed. Run: pip install scanpy")
    print("Showing simulated workflow below.")

def simulate_scrna(n_cells: int = 800, n_genes: int = 2000,
                   n_celltypes: int = 4, seed: int = 42) -> dict:
    """Simulate scRNA-seq count matrix with known cell types."""    rng = np.random.default_rng(seed)
    celltype_names = ["Cancer cell", "T cell", "Macrophage", "Fibroblast"]
    cells_per_type = n_cells // n_celltypes
    labels = []
    for ct in celltype_names:
        labels.extend([ct] * cells_per_type)

    # Each cell type has characteristic gene expression
    X = rng.negative_binomial(3, 0.3, size=(n_cells, n_genes)).astype(float)
    for i, ct in enumerate(celltype_names):
        idx_cells = range(i*cells_per_type, (i+1)*cells_per_type)
        idx_genes = range(i*50, (i+1)*50)   # marker genes
        for c in idx_cells:
            for g in idx_genes:
                X[c, g] += rng.poisson(20)  # high expression for markers

    genes = [f"Gene_{i:04d}" for i in range(n_genes)]
    cells = [f"Cell_{i:04d}" for i in range(n_cells)]
    return {"X": X, "genes": genes, "cells": cells, "labels": np.array(labels),
            "celltype_names": celltype_names}

sc_data = simulate_scrna()
print(f"Simulated scRNA: {sc_data['X'].shape[0]} cells × {sc_data['X'].shape[1]} genes")
print(f"Cell types: {dict(zip(*np.unique(sc_data['labels'], return_counts=True)))}")

# ── QC metrics ────────────────────────────────────────────────────────────────
n_counts = sc_data['X'].sum(axis=1)
n_genes_per_cell = (sc_data['X'] > 0).sum(axis=1)
pct_mito = np.random.uniform(0.02, 0.25, len(sc_data['cells']))  # simulated

print(f"\nQC Metrics:")
print(f"  Total counts/cell: median={np.median(n_counts):.0f}  IQR=[{np.percentile(n_counts,25):.0f}, {np.percentile(n_counts,75):.0f}]")
print(f"  Genes/cell:        median={np.median(n_genes_per_cell):.0f}")
print(f"  Mito %:            median={np.median(pct_mito)*100:.1f}%")

# Standard QC filter
keep = (n_counts > np.percentile(n_counts, 5)) & (pct_mito < 0.2)
print(f"  After QC filter:   {keep.sum()}/{len(keep)} cells retained")

In [ ]:
# ── 2.2 Normalisation, PCA, UMAP ─────────────────────────────────────────────
from sklearn.decomposition import PCA
try:
    from umap import UMAP
    UMAP_AVAIL = True
except ImportError:
    from sklearn.manifold import TSNE
    UMAP_AVAIL = False

# 1. Filter & normalise (log1p + library-size normalisation)
X_filt = sc_data['X']
labels = sc_data['labels']

# Normalise to 10,000 counts per cell (scran/Seurat standard)
X_norm = X_filt / X_filt.sum(axis=1, keepdims=True) * 10_000
X_log  = np.log1p(X_norm)   # log1p stabilises variance

# 2. HVG selection (top 500 most variable genes)
gene_var   = X_log.var(axis=0)
hvg_idx    = np.argsort(gene_var)[-500:]
X_hvg      = X_log[:, hvg_idx]

# 3. PCA
pca     = PCA(n_components=30, random_state=42)
X_pca   = pca.fit_transform(X_hvg)
var_exp = pca.explained_variance_ratio_

# 4. Embedding (UMAP or t-SNE)
if UMAP_AVAIL:
    reducer = UMAP(n_components=2, n_neighbors=20, min_dist=0.3, random_state=42)
    embed   = reducer.fit_transform(X_pca)
    method  = "UMAP"
else:
    reducer = TSNE(n_components=2, random_state=42, perplexity=30)
    embed   = reducer.fit_transform(X_pca[:, :10])
    method  = "t-SNE"

# 5. Visualise
COLOURS = {"Cancer cell":"#E74C3C","T cell":"#1565C0",
           "Macrophage":"#27AE60","Fibroblast":"#E67E22"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Elbow plot
ax = axes[0]
ax.plot(range(1,16), var_exp[:15]*100, 'o-', color='#1565C0', lw=2.2, ms=6)
ax.set_xlabel("Principal Component"); ax.set_ylabel("% Variance Explained")
ax.set_title("Elbow Plot — PCA", fontweight="bold")
ax.axvline(10, color="red", linestyle="--", alpha=0.5, label="Chosen PCs")
ax.legend(); ax.grid(True, alpha=0.3)

# UMAP/t-SNE coloured by cell type
ax = axes[1]
for ct in sc_data['celltype_names']:
    mask = labels == ct
    ax.scatter(embed[mask,0], embed[mask,1], c=COLOURS[ct],
               s=8, alpha=0.7, label=ct)
ax.set_title(f"{method} — Cell Types", fontweight="bold")
ax.set_xlabel(f"{method} 1"); ax.set_ylabel(f"{method} 2")
ax.legend(fontsize=9, markerscale=3); ax.grid(True, alpha=0.25)

plt.tight_layout(); plt.show()
print(f"scRNA {method} plot done ✓")

---
## Section 3 — Genomics: Mutation Signatures

In [ ]:
# ── 3.1 COSMIC mutational signatures ─────────────────────────────────────────
# Mutational signatures decompose somatic mutations into known processes.
# SBS3 = HRD (homologous recombination deficiency) → PARP inhibitor sensitivity

# 96-channel trinucleotide mutation spectrum (simplified to 6 for illustration)
CONTEXTS = ["C>A", "C>G", "C>T", "T>A", "T>C", "T>G"]

# Known COSMIC SBS signatures (simplified 6-channel versions)
SBS_SIGNATURES = {
    "SBS1":  np.array([0.01, 0.01, 0.30, 0.05, 0.10, 0.03]),  # deamination (aging)
    "SBS2":  np.array([0.20, 0.01, 0.05, 0.01, 0.01, 0.02]),  # APOBEC
    "SBS3":  np.array([0.15, 0.12, 0.12, 0.16, 0.14, 0.13]),  # HRD — uniform
    "SBS4":  np.array([0.25, 0.03, 0.05, 0.15, 0.02, 0.02]),  # smoking
    "SBS13": np.array([0.03, 0.30, 0.03, 0.02, 0.02, 0.10]),  # APOBEC (C>G)
}

# Normalize each signature
for k in SBS_SIGNATURES:
    SBS_SIGNATURES[k] /= SBS_SIGNATURES[k].sum()

def decompose_spectrum(observed: np.ndarray,
                       signatures: dict,
                       n_iter: int = 500) -> dict:
    """
    Non-negative least squares decomposition of mutation spectrum
    into known COSMIC signatures. Returns attribution weights.
    """
    from scipy.optimize import nnls
    sig_matrix = np.column_stack(list(signatures.values()))
    weights, residual = nnls(sig_matrix, observed)
    total = weights.sum()
    return {name: float(w/total) if total > 0 else 0
            for name, w in zip(signatures.keys(), weights)}

# Simulate two tumour samples
np.random.seed(42)
BRCA_sample  = (np.array([0.12, 0.08, 0.15, 0.14, 0.12, 0.10]) +
                np.random.dirichlet([2]*6)*0.3)  # HRD-dominated
Lung_sample  = (np.array([0.30, 0.04, 0.06, 0.18, 0.03, 0.03]) +
                np.random.dirichlet([2]*6)*0.3)  # smoking-dominated
BRCA_sample /= BRCA_sample.sum()
Lung_sample /= Lung_sample.sum()

print("Signature decomposition:")
for sample_name, spectrum in [("BRCA (HRD)", BRCA_sample), ("Lung (Smoker)", Lung_sample)]:
    attrib = decompose_spectrum(spectrum, SBS_SIGNATURES)
    print(f"\n  {sample_name}:")
    for sig, weight in sorted(attrib.items(), key=lambda x: -x[1]):
        bar = "█" * int(weight * 30)
        print(f"    {sig}: {weight:.3f}  {bar}")

    sbs3 = attrib.get("SBS3", 0)
    if sbs3 > 0.2:
        print(f"    → SBS3={sbs3:.2f}: HIGH HRD — PARP inhibitor ELIGIBLE")
    else:
        print(f"    → SBS3={sbs3:.2f}: Low HRD signal")

---
## Section 4 — Proteomics: Differential Protein Expression

In [ ]:
# ── 4.1 TMT/LFQ proteomics workflow ──────────────────────────────────────────
# Simulate a CPTAC-style mass spectrometry protein abundance matrix

def simulate_proteomics(n_proteins: int = 3000, n_samples: int = 12,
                        n_de_proteins: int = 300, seed: int = 42):
    """Simulate log2 protein intensity matrix (TMT/LFQ)."""    rng = np.random.default_rng(seed)
    groups = ["CTRL"]*6 + ["TREAT"]*6
    proteins = [f"Prot_{i:04d}" for i in range(n_proteins)]
    samples  = [f"{g}_{i%6+1}" for i, g in enumerate(groups)]

    # Base intensities (log2-normal)
    base = rng.normal(25, 3, n_proteins)

    # Simulate with DE proteins and 10% missing values
    X = np.zeros((n_proteins, n_samples))
    de_proteins = proteins[:n_de_proteins]
    for j in range(n_samples):
        treated = groups[j] == "TREAT"
        for i in range(n_proteins):
            fc = 0
            if treated and proteins[i] in de_proteins:
                fc = rng.choice([-1.5, -1, 0.5, 1, 1.5])
            X[i, j] = base[i] + fc + rng.normal(0, 0.5)

    # Add 10% missing (MNAR — missing-not-at-random, common in proteomics)
    mask = rng.random((n_proteins, n_samples)) < 0.10
    X[mask] = np.nan

    df = pd.DataFrame(X, index=proteins, columns=samples)
    meta = pd.DataFrame({"group": groups}, index=samples)
    return df, meta, de_proteins

prot_df, prot_meta, true_de_prot = simulate_proteomics()

# ── Imputation ────────────────────────────────────────────────────────────────
# MinProb imputation: replace NaN with small value (MNAR assumption)
def minprob_impute(df: pd.DataFrame, width: float = 0.3, downshift: float = 1.8):
    """Impute missing values with Gaussian distribution at low intensity."""    df_imp = df.copy()
    for col in df.columns:
        missing = df[col].isna()
        if missing.any():
            col_mean = df[col].quantile(0.01)
            col_std  = df[col].std(skipna=True)
            fill_vals = np.random.normal(
                col_mean - downshift*col_std,
                width*col_std,
                missing.sum()
            )
            df_imp.loc[missing, col] = fill_vals
    return df_imp

prot_imputed = minprob_impute(prot_df)
missing_pct  = prot_df.isna().mean().mean() * 100
print(f"Proteomics matrix: {prot_df.shape}")
print(f"Missing values: {missing_pct:.1f}% → imputed with MinProb")

# ── Differential protein expression ───────────────────────────────────────────
ctrl_cols  = prot_meta[prot_meta["group"]=="CTRL"].index.tolist()
treat_cols = prot_meta[prot_meta["group"]=="TREAT"].index.tolist()

prot_results = []
for prot in prot_imputed.index:
    ctrl_vals  = prot_imputed.loc[prot, ctrl_cols].values
    treat_vals = prot_imputed.loc[prot, treat_cols].values
    log2fc = treat_vals.mean() - ctrl_vals.mean()
    _, pval = stats.ttest_ind(treat_vals, ctrl_vals)
    prot_results.append({"protein": prot, "log2FC": log2fc, "pval": pval})

prot_de = pd.DataFrame(prot_results)
from statsmodels.stats.multitest import multipletests
_, padj, _, _ = multipletests(prot_de["pval"].fillna(1), method="fdr_bh")
prot_de["padj"] = padj
prot_de["sig"]  = (prot_de["padj"] < 0.05) & (prot_de["log2FC"].abs() > 0.5)

print(f"Significant DE proteins: {prot_de['sig'].sum()}")
print(prot_de[prot_de["sig"]].head(5)[["protein","log2FC","padj"]].to_string())

---
## Section 5 — Multi-Omics Integration (MOFA+)

In [ ]:
# ── 5.1 MOFA+ style multi-omics factor analysis ──────────────────────────────
# MOFA+ (Argelaguet 2020) finds latent factors shared and unique across omics.
# pip install mofax  (Python bindings)
# R: BiocManager::install('MOFA2')

# We implement a simplified NMF-based multi-omics integration here
# that captures the core idea of MOFA+.

from sklearn.decomposition import NMF
from scipy.stats import spearmanr

def integrate_omics_nmf(omics_dict: dict, n_factors: int = 10,
                         seed: int = 42) -> dict:
    """
    Simple multi-omics integration using concatenated NMF.
    In production: use MOFA+ (much more principled — handles missing data,
    views per modality, variational inference).

    Parameters
    ----------
    omics_dict : {name: np.ndarray [n_samples × n_features]}
    n_factors  : number of latent factors

    Returns
    -------
    factors    : [n_samples × n_factors]  shared latent space
    loadings   : per-modality feature weights
    """
    # Concatenate all omics (after standardisation)
    X_parts, feature_labels = [], []
    for name, X in omics_dict.items():
        # Min-max normalise to [0,1] for NMF (requires non-negative)
        X_pos = X - X.min(axis=0, keepdims=True)
        X_pos /= (X_pos.max(axis=0, keepdims=True) + 1e-8)
        X_parts.append(X_pos)
        feature_labels.extend([f"{name}_{i}" for i in range(X.shape[1])])

    X_concat = np.hstack(X_parts)

    # NMF decomposition: X ≈ W × H
    model     = NMF(n_components=n_factors, init='nndsvda',
                    random_state=seed, max_iter=500)
    W         = model.fit_transform(X_concat)   # [n_samples × n_factors]
    H         = model.components_               # [n_factors × n_features]

    return {"factors": W, "loadings": H,
            "feature_labels": feature_labels,
            "reconstruction_error": model.reconstruction_err_}

# ── Build multi-omics dataset ──────────────────────────────────────────────────
# Use same sample IDs across all omics!
N_SAMPLES = 40

# Transcriptomics: top 100 DE genes
rnaseq_data = counts.T.values[:N_SAMPLES, :100].astype(float)
rnaseq_data = np.log1p(rnaseq_data)

# Proteomics: top 50 proteins
prot_data   = prot_imputed.T.values[:N_SAMPLES, :50].astype(float)

# Clinical: age, stage, BMI (simulated)
np.random.seed(42)
clinical    = np.column_stack([
    np.random.randint(40, 80, N_SAMPLES),  # age
    np.random.randint(1,  5,  N_SAMPLES),  # stage
    np.random.uniform(18, 35, N_SAMPLES),  # BMI
])

omics_dict = {
    "RNAseq":    rnaseq_data,
    "Proteomics": prot_data,
    "Clinical":   clinical,
}

print("Multi-omics input:")
for name, X in omics_dict.items():
    print(f"  {name:12s}: {X.shape[0]} samples × {X.shape[1]} features")

result   = integrate_omics_nmf(omics_dict, n_factors=5)
factors  = result["factors"]
print(f"\nMOFA+ factors: {factors.shape}")
print(f"Reconstruction error: {result['reconstruction_error']:.2f}")

In [ ]:
# ── 5.2 Visualise factors and associate with phenotype ─────────────────────────
np.random.seed(42)
# Simulate binary outcome (drug response)
response = (factors[:, 0] + np.random.normal(0, 0.3, N_SAMPLES) > factors[:,0].mean()).astype(int)

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Factor correlation matrix
ax1 = fig.add_subplot(gs[0, 0])
corr_matrix = np.corrcoef(factors.T)
im = ax1.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax1.set_xticks(range(5)); ax1.set_xticklabels([f"F{i+1}" for i in range(5)])
ax1.set_yticks(range(5)); ax1.set_yticklabels([f"F{i+1}" for i in range(5)])
ax1.set_title("Factor Correlation Matrix", fontweight="bold")
plt.colorbar(im, ax=ax1)

# Scatter: Factor 1 vs Factor 2, coloured by response
ax2 = fig.add_subplot(gs[0, 1])
for val, colour, label in [(0, '#1565C0', 'Non-responder'), (1, '#E74C3C', 'Responder')]:
    m = response == val
    ax2.scatter(factors[m, 0], factors[m, 1], c=colour, s=40, alpha=0.7, label=label)
ax2.set_xlabel("Factor 1"); ax2.set_ylabel("Factor 2")
ax2.set_title("Factor 1 vs 2 — Drug Response", fontweight="bold")
ax2.legend(); ax2.grid(True, alpha=0.3)

# Factor-phenotype association (correlation per factor)
ax3 = fig.add_subplot(gs[0, 2])
correlations = [spearmanr(factors[:, f], response).statistic for f in range(5)]
colours = ['#E74C3C' if abs(c) > 0.3 else '#1565C0' for c in correlations]
ax3.bar([f"F{i+1}" for i in range(5)], [abs(c) for c in correlations],
        color=colours)
ax3.set_ylabel("|Spearman ρ| with response")
ax3.set_title("Factor-Phenotype Association", fontweight="bold")
ax3.axhline(0.3, color='k', linestyle='--', lw=1, label='|ρ|=0.3 threshold')
ax3.legend(); ax3.grid(True, alpha=0.3, axis='y')

# Factor loading: which omics features drive Factor 1?
ax4 = fig.add_subplot(gs[1, :])
loadings_f1 = result["loadings"][0, :]
# Top 20 features by absolute loading
top_idx  = np.argsort(np.abs(loadings_f1))[-20:]
top_vals = loadings_f1[top_idx]
top_labs = [result["feature_labels"][i] for i in top_idx]
bar_cols  = ['#E74C3C' if v > 0 else '#1565C0' for v in top_vals]
ax4.barh(range(20), top_vals, color=bar_cols)
ax4.set_yticks(range(20)); ax4.set_yticklabels(top_labs, fontsize=8)
ax4.axvline(0, color='k', lw=0.8)
ax4.set_xlabel("Loading weight"); ax4.set_title("Factor 1 Top Feature Loadings", fontweight="bold")
ax4.grid(True, alpha=0.3, axis='x')

plt.suptitle("Multi-Omics Integration (MOFA+ / NMF)", fontsize=14, fontweight="bold")
plt.savefig("multiomics_integration.png", dpi=130, bbox_inches="tight")
plt.show()
print("Integration plots saved ✓")

---
## Section 6 — Toxicogenomics

Gene expression changes as biomarkers of toxicity — connecting molecular perturbations to organ-level injury.

In [ ]:
# ── 6.1 Toxicogenomics: DILI gene signature scoring ──────────────────────────
# Drug-induced liver injury (DILI) has characteristic transcriptional signatures.
# We score each compound's expression profile against known DILI signatures.

# Known DILI gene signatures (simplified)
DILI_SIGNATURES = {
    "Mitochondrial_stress": [
        "Gene_0001", "Gene_0003", "Gene_0005", "Gene_0007", "Gene_0009",
        "Gene_0011", "Gene_0013", "Gene_0015", "Gene_0017", "Gene_0019",
    ],
    "Oxidative_stress": [
        "Gene_0021", "Gene_0023", "Gene_0025", "Gene_0027", "Gene_0029",
        "Gene_0031", "Gene_0033", "Gene_0035", "Gene_0037", "Gene_0039",
    ],
    "Cholestasis": [
        "Gene_0041", "Gene_0043", "Gene_0045", "Gene_0047", "Gene_0049",
        "Gene_0051", "Gene_0053", "Gene_0055", "Gene_0057", "Gene_0059",
    ],
    "Hepatocyte_death": [
        "Gene_0061", "Gene_0063", "Gene_0065", "Gene_0067", "Gene_0069",
        "Gene_0071", "Gene_0073", "Gene_0075", "Gene_0077", "Gene_0079",
    ],
}

def score_signature(expression_profile: pd.Series,
                    signature_genes: list,
                    method: str = "ssgsea") -> float:
    """
    Score a gene expression profile against a gene set using mean z-score.
    For production, use gseapy.ssgsea() or scanpy.tl.score_genes().
    """
    valid = [g for g in signature_genes if g in expression_profile.index]
    if not valid:
        return 0.0
    # Z-score the entire profile
    mu, sigma = expression_profile.mean(), expression_profile.std()
    if sigma == 0:
        return 0.0
    z = (expression_profile - mu) / sigma
    return float(z[valid].mean())

# Score all treatment samples against DILI signatures
treat_samples = [c for c in counts.columns if c.startswith("TREAT")]
log_counts    = np.log1p(counts[treat_samples])

sig_scores = {}
for sig_name, sig_genes in DILI_SIGNATURES.items():
    scores = {}
    for sample in treat_samples:
        scores[sample] = score_signature(log_counts[sample], sig_genes)
    sig_scores[sig_name] = scores

sig_df = pd.DataFrame(sig_scores)
print("DILI Signature Scores (treated samples):")
print(sig_df.round(3).to_string())

# Flag samples with high DILI risk
threshold = 0.5
high_risk = sig_df[sig_df.max(axis=1) > threshold]
print(f"\nHigh DILI risk samples (any score > {threshold}): {len(high_risk)}")
for sample, row in high_risk.iterrows():
    top_sig = row.idxmax()
    print(f"  {sample}: {top_sig} score = {row[top_sig]:.3f}")

# Heatmap
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(sig_df.T.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(treat_samples))); ax.set_xticklabels(treat_samples, rotation=45, ha='right')
ax.set_yticks(range(len(DILI_SIGNATURES))); ax.set_yticklabels(list(DILI_SIGNATURES.keys()))
plt.colorbar(im, ax=ax, label='ssGSEA score')
ax.set_title('DILI Toxicogenomic Signature Scores', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.2 Pathway enrichment (simplified ORA) ──────────────────────────────────
# Over-Representation Analysis: are DE genes enriched in known pathways?

# Minimal pathway database
PATHWAYS = {
    "Apoptosis":              set(de_results[de_results["sig"] & (de_results["log2FC"] < 0)]["gene"].head(30)),
    "Cell_proliferation":     set(de_results[de_results["sig"] & (de_results["log2FC"] > 0)]["gene"].head(30)),
    "Oxidative_stress":       set([f"Gene_{i:04d}" for i in range(0, 30)]),
    "DNA_damage_response":    set([f"Gene_{i:04d}" for i in range(30, 60)]),
    "Metabolic_activation":   set([f"Gene_{i:04d}" for i in range(60, 90)]),
    "Inflammatory_response":  set([f"Gene_{i:04d}" for i in range(90, 120)]),
}

def ora_enrichment(de_genes: set, pathway_genes: set,
                   universe_size: int = 5000) -> tuple[float, float]:
    """
    Fisher's exact test for over-representation.
    Returns (odds_ratio, p_value).
    """
    from scipy.stats import fisher_exact
    a = len(de_genes & pathway_genes)         # DE genes in pathway
    b = len(de_genes) - a                      # DE genes not in pathway
    c = len(pathway_genes) - a                 # non-DE genes in pathway
    d = universe_size - a - b - c              # non-DE, not in pathway
    _, pval = fisher_exact([[a, b], [c, d]], alternative='greater')
    or_val  = (a * d) / (b * c) if (b * c) > 0 else np.inf
    return round(or_val, 2), pval

# Run ORA
de_gene_set = set(de_results[de_results["sig"]]["gene"])
print(f"ORA: {len(de_gene_set)} DE genes vs {len(PATHWAYS)} pathways")
print(f"{'Pathway':25s} {'OR':>6} {'p-value':>10} {'Sig':>5}")
print("-" * 55)

ora_results = []
for pathway, genes in PATHWAYS.items():
    or_val, pval = ora_enrichment(de_gene_set, genes)
    ora_results.append({"pathway": pathway, "OR": or_val, "pval": pval})

from statsmodels.stats.multitest import multipletests
ora_df = pd.DataFrame(ora_results)
_, ora_df["padj"], _, _ = multipletests(ora_df["pval"], method="fdr_bh")

for _, row in ora_df.sort_values("padj").iterrows():
    sig = "***" if row.padj<0.001 else "**" if row.padj<0.01 else "*" if row.padj<0.05 else ""
    print(f"{row.pathway:25s} {row.OR:6.2f} {row.padj:10.4f} {sig:>5}")

In [ ]:
# ── 6.3 Multi-omics cheatsheet ────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║            Multi-Omics for Drug Discovery — Quick Reference             ║
╠══════════════════════════════════════════════════════════════════════════╣
║ WORKFLOWS                                                                ║
║  RNA-seq      Count matrix → DESeq2 → volcano → GSEA                   ║
║  scRNA-seq    10x → Scanpy → QC → norm → HVG → PCA → UMAP → Leiden    ║
║  Proteomics   TMT/LFQ → impute → limma/t-test → PPI                    ║
║  Genomics     MAF → SBS signatures → COSMIC → actionability            ║
║  Metabolomics LC-MS → normalise → PLSDA → pathway                      ║
║  Integration  MOFA+ / SNF / deep learning (CGAE, MIRA)                ║
╠══════════════════════════════════════════════════════════════════════════╣
║ KEY TOOLS                                                                ║
║  pydeseq2         Differential expression (Python DESeq2)              ║
║  scanpy           Single-cell RNA-seq analysis                         ║
║  gseapy           GSEA, ssGSEA, ORA enrichment                        ║
║  mofax            MOFA+ Python bindings                                ║
║  statsmodels      FDR correction, linear models                        ║
║  scipy.stats      t-test, Mann-Whitney, Fisher's exact                 ║
║  sklearn          PCA, NMF, clustering                                  ║
╠══════════════════════════════════════════════════════════════════════════╣
║ CRITICAL RULES                                                           ║
║  ✓ Always correct for multiple testing (FDR-BH or Bonferroni)         ║
║  ✓ Use log2FC > 1 AND padj < 0.05 for DE calls (not just p-value!)    ║
║  ✓ Check for batch effects before integration (PCA colour by batch)   ║
║  ✓ Match sample IDs EXACTLY across omics layers                        ║
║  ✓ scRNA: QC on n_counts, n_genes, % mito before any analysis         ║
║  ✓ Proteomics: impute missing values (MNAR ≠ MCAR)                    ║
║  ✓ Signature scoring requires z-scored expression (not raw counts)     ║
╠══════════════════════════════════════════════════════════════════════════╣
║ TOXICOLOGY-SPECIFIC                                                      ║
║  DILI signatures    Mitochondrial / oxidative / cholestasis pathways   ║
║  ToxCast            High-throughput transcriptomic assay data          ║
║  LINCS L1000        1000-gene expression landmarks                     ║
║  DrugMatrix         NHEERL rat transcriptomic toxicology DB            ║
║  MicrotoxDB         In vitro toxicogenomics database                   ║
╚══════════════════════════════════════════════════════════════════════════╝
""")